# Abalone: model comparison and evaluation

Predict `rings` with a simple, reproducible workflow. Add 1.5 to a ring prediction to express estimated age in years;
the constant shift does not change RMSE or MAE.

**Method:** reserve 20% as a test set; compare a mean baseline and nine predictive candidates with five-fold
cross-validation on the remaining 80%. Select the candidate with the lowest mean validation RMSE, then evaluate it once on the test set.

Source: [Kaggle Abalone dataset](https://www.kaggle.com/datasets/rodolfomendes/abalone-dataset), `data/abalone.csv`.
Use Python 3.11 and the project `.venv` kernel.

**Verified reference run** (supplied CSV, seed 42): RBF SVR has the lowest mean CV RMSE, **2.126 ± 0.139** rings (fold SD).
Its reference test RMSE is **2.218**, versus **3.378** for the mean baseline. Small CV differences do not establish a universal winner.

## 1. Setup and assumptions

Use the same positive-height population and split as `eda.ipynb`. Rows are assumed independent; no farm, animal,
or collection-time grouping identifiers are available. This random-split evaluation does not measure performance under future distribution shifts.

Run cells in order from the repository root or `notebooks/`. No hyperparameter search or test-driven tuning is performed.

In [ ]:
import hashlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import RegressorMixin, clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 12, "axes.labelsize": 10})
pd.set_option("display.max_columns", 12)
pd.set_option("display.precision", 3)

In [ ]:
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), Path.cwd().parent) if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start the notebook from the repository root or its notebooks/ directory.")

DATA_PATH = PROJECT_ROOT / "data" / "abalone.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Download the Kaggle Abalone CSV and save it as {DATA_PATH}.")

TARGET = "rings"
CATEGORICAL_COLS = ["sex"]
NUMERICAL_COLS = ["length", "diameter", "height", "whole_weight", "shucked_weight", "viscera_weight", "shell_weight"]
FEATURE_COLS = CATEGORICAL_COLS + NUMERICAL_COLS
TEST_SIZE = 0.2
RANDOM_STATE = 42
DATA_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()


N_SPLITS = 5

## 2. Load, validate, and prepare the population

Validate the expected schema instead of silently passing missing or nonfinite measurements to the estimator.
Apply only the fixed `height > 0` eligibility rule; do not estimate outlier cutoffs from the test set.
The small functions below can be moved into modules in Stage 2.

In [ ]:
def load_data(path: Path) -> pd.DataFrame:
    """Read the expected Abalone schema and retain positive-height records."""
    frame = pd.read_csv(path).rename(columns=lambda name: name.strip().lower().replace(" ", "_"))
    missing = set(FEATURE_COLS + [TARGET]) - set(frame.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    if not frame.columns.is_unique:
        raise ValueError("Column names must be unique after normalization.")
    if frame[FEATURE_COLS + [TARGET]].isna().any().any():
        raise ValueError("Missing input values require an explicit imputation policy.")
    if not frame["sex"].isin(["F", "I", "M"]).all():
        raise ValueError("Expected sex values F, I, or M.")
    numeric = frame[NUMERICAL_COLS + [TARGET]].to_numpy(dtype=float)
    if not np.isfinite(numeric).all():
        raise ValueError("Numerical inputs and target must be finite.")
    if not ((frame[TARGET] > 0) & (frame[TARGET] % 1 == 0)).all():
        raise ValueError("Rings must be positive integers.")
    return frame.loc[frame["height"] > 0, FEATURE_COLS + [TARGET]].copy()


def extract_features(frame: pd.DataFrame) -> list[dict]:
    """Build predictor dictionaries without including the target."""
    return frame[FEATURE_COLS].to_dict(orient="records")


def make_pipeline(estimator: RegressorMixin) -> Pipeline:
    """Fit encoding and any required standardization inside each cross-validation fold."""
    scale_sensitive = (LinearRegression, Ridge, Lasso, ElasticNet, KNeighborsRegressor, SVR)
    scaler = StandardScaler() if isinstance(estimator, scale_sensitive) else "passthrough"
    return Pipeline([("vectorizer", DictVectorizer(sparse=False)), ("scaler", scaler), ("regressor", clone(estimator))])


def evaluate_model(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """Report errors in rings and the coefficient of determination."""
    return {
        "rmse": float(root_mean_squared_error(y_true, y_pred)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }

In [ ]:
data = load_data(DATA_PATH)
development_df, test_df = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE)
development_records = extract_features(development_df)
y_development = development_df[TARGET].to_numpy()
print(f"Eligible rows: {len(data):,}; development: {len(development_df):,}; test: {len(test_df):,}")
print(f"Input SHA-256: {DATA_SHA256}")
development_df.head()

## 3. Compare models on development data

Use the same five shuffled folds for every candidate. Encoding and any standardization are fitted only on each fold's
training portion, so validation rows do not influence the learned transformations. The mean baseline is also fitted in each fold.

| Family | Candidate | Purpose |
|---|---|---|
| Baseline | Training mean | Minimum reference for predictive value |
| Linear | Ordinary least squares | Simple additive relationship |
| Regularized linear | Ridge | L2 shrinkage for correlated measurements |
| Regularized linear | Lasso | L1 shrinkage and possible coefficient sparsity |
| Regularized linear | Elastic Net | Combined L1/L2 shrinkage |
| Neighbors | Distance-weighted KNN | Local nonlinear relationships |
| Kernel | RBF SVR | Smooth nonlinear relationships |
| Tree ensemble | Random forest | Bagged decision trees |
| Tree ensemble | Extra Trees | More randomized tree splits |
| Tree ensemble | Gradient boosting | Sequential correction of residuals |

Standardize encoded features for linear, regularized, KNN, and SVR models. This includes the one-hot indicators;
scaling changes their relative contribution to distance and penalty terms without imposing an order on `sex`.
Tree models and the baseline use unscaled features. Dense encoding is practical here because there are only ten encoded columns.

**Selection rule fixed before evaluation:** among the nine predictive candidates, choose the lowest mean validation RMSE.
Each model uses one visible, fixed parameter configuration; this is a model-family comparison, not an exhaustive tuning exercise.
CV scores used for selection can favor a chance winner, so small differences do not establish a universally superior model.
Fold SD describes variability across the five folds and is not a confidence interval. Fit times are machine-dependent.


Distance-weighted KNN can have zero training error because a training row is its own nearest neighbor.
That is memorization, not evidence of perfect predictive accuracy; use its validation scores for comparison.

In [ ]:
candidates = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Linear regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01, max_iter=20000),
    "Elastic Net": ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=20000),
    "KNN": KNeighborsRegressor(n_neighbors=15, weights="distance", n_jobs=1),
    "RBF SVR": SVR(kernel="rbf", C=10.0, epsilon=0.1, gamma="scale"),
    "Random forest": RandomForestRegressor(n_estimators=100, max_depth=10, random_state=RANDOM_STATE, n_jobs=1),
    "Extra Trees": ExtraTreesRegressor(n_estimators=200, max_depth=15, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=1),
    "Gradient boosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=2, random_state=RANDOM_STATE),
}
cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
scoring = {"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error", "r2": "r2"}

In [ ]:
def cross_validate_model(name: str, estimator: RegressorMixin) -> dict:
    """Compare a candidate using development folds without reading the held-out test targets."""
    scores = cross_validate(
        make_pipeline(estimator),
        development_records,
        y_development,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=1,
        error_score="raise",
    )
    return {
        "model": name,
        "cv_rmse_mean": float(np.mean(-scores["test_rmse"])),
        "cv_rmse_std": float(np.std(-scores["test_rmse"], ddof=1)),
        "cv_mae_mean": float(np.mean(-scores["test_mae"])),
        "cv_r2_mean": float(np.mean(scores["test_r2"])),
        "train_rmse_mean": float(np.mean(-scores["train_rmse"])),
        "fit_seconds_mean": float(np.mean(scores["fit_time"])),
    }


comparison = pd.DataFrame([cross_validate_model(name, estimator) for name, estimator in candidates.items()])
comparison = comparison.sort_values("cv_rmse_mean").reset_index(drop=True)
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(11, max(5, 0.55 * len(comparison))), constrained_layout=True)
positions = np.arange(len(comparison))
ax.barh(positions - 0.18, comparison["train_rmse_mean"], height=0.35, color="#999999", label="Training mean")
ax.barh(
    positions + 0.18,
    comparison["cv_rmse_mean"],
    height=0.35,
    xerr=comparison["cv_rmse_std"],
    color="#0072B2",
    capsize=3,
    label="Validation mean ± fold SD",
)
ax.set_yticks(positions, comparison["model"])
ax.invert_yaxis()
ax.set(title="Five-fold development comparison: lower RMSE is better", xlabel="RMSE (rings)", ylabel="Model")
ax.grid(axis="x", alpha=0.2)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.show()

best_row = comparison.loc[comparison["model"] != "Mean baseline"].iloc[0]
selected_name = best_row["model"]
print(f"Selected using development CV only: {selected_name}")
print(f"Mean validation RMSE: {best_row['cv_rmse_mean']:.3f} ± {best_row['cv_rmse_std']:.3f} rings (fold SD)")

## 4. Final test evaluation

Refit the selected pipeline on every development row, then evaluate the reserved test set. Fit the mean baseline
on development data as a reference. Model choice is already fixed; the following diagnostics must not be used to retune this version.

The fitted pipeline contains both the encoder and regressor. It accepts feature dictionaries with normalized column names.

In [ ]:
selected_model = make_pipeline(candidates[selected_name]).fit(development_records, y_development)
baseline_model = make_pipeline(candidates["Mean baseline"]).fit(development_records, y_development)
test_records = extract_features(test_df)
y_test = test_df[TARGET].to_numpy()
test_predictions = selected_model.predict(test_records)
baseline_predictions = baseline_model.predict(test_records)
train_metrics = evaluate_model(y_development, selected_model.predict(development_records))
test_metrics = evaluate_model(y_test, test_predictions)
baseline_metrics = evaluate_model(y_test, baseline_predictions)

evaluation = pd.DataFrame(
    [train_metrics, test_metrics, baseline_metrics],
    index=[f"{selected_name}: development fit", f"{selected_name}: held-out test", "Mean baseline: held-out test"],
)
evaluation

## 5. Diagnose held-out errors

Error is **prediction minus actual rings**: negative values mean underprediction. The identity line in the first plot
represents perfect prediction. Report group sizes alongside errors so sparse groups are not mistaken for equally precise estimates.

In [ ]:
errors = test_predictions - y_test
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
limits = [min(y_test.min(), test_predictions.min()), max(y_test.max(), test_predictions.max())]
axes[0].scatter(y_test, test_predictions, alpha=0.35, s=15, color="#0072B2")
axes[0].plot(limits, limits, "--", color="#D55E00", label="Perfect prediction")
axes[0].set(title="Actual vs predicted: held-out test", xlabel="Actual rings", ylabel="Predicted rings")
axes[0].legend()
axes[1].scatter(test_predictions, errors, alpha=0.35, s=15, color="#0072B2")
axes[1].axhline(0, color="#D55E00", linestyle="--")
axes[1].set(title="Prediction errors: held-out test", xlabel="Predicted rings", ylabel="Predicted − actual rings")
axes[2].hist(errors, bins=30, color="#0072B2", edgecolor="white")
axes[2].axvline(0, color="#D55E00", linestyle="--")
axes[2].set(title="Error distribution: held-out test", xlabel="Predicted − actual rings", ylabel="Abalones")
plt.show()

In [ ]:
diagnostics = pd.DataFrame({"actual_rings": y_test, "predicted_rings": test_predictions, "error": errors})
diagnostics["ring_group"] = pd.cut(y_test, bins=[-np.inf, 7, 11, np.inf], labels=["1–7", "8–11", "12+"])
group_rows = []
for name, group in diagnostics.groupby("ring_group", observed=True):
    group_rows.append(
        {
            "actual_ring_group": name,
            "count": len(group),
            "rmse": float(np.sqrt(np.mean(group["error"] ** 2))),
            "mae": float(group["error"].abs().mean()),
            "mean_error": float(group["error"].mean()),
        }
    )
error_by_group = pd.DataFrame(group_rows)
error_by_group

Use this table to check whether error and bias differ between ring groups. Negative mean error indicates
underprediction; positive mean error indicates overprediction. Group-specific estimates with fewer rows are less stable.
These are descriptive diagnostics after selection, not a reason to retune the current model using the test set.

## 6. Verify target-free inference

Pass only feature columns to the fitted pipeline and check that its predictions match those used for evaluation.
Convert the predicted ring counts to estimated age by adding 1.5.

In [ ]:
inference_records = extract_features(test_df.drop(columns=[TARGET]))
inference_predictions = selected_model.predict(inference_records)
np.testing.assert_allclose(inference_predictions, test_predictions)
print("Target-free inference check passed.")
pd.DataFrame({"predicted_rings": inference_predictions[:5], "estimated_age_years": inference_predictions[:5] + 1.5})

## 7. Results and handoff

The summary below is computed from this execution, rather than copied from an earlier run.

In [ ]:
improvement = 100 * (1 - test_metrics["rmse"] / baseline_metrics["rmse"])
print(f"Selected model: {selected_name}")
print(f"Held-out RMSE: {test_metrics['rmse']:.3f} rings; MAE: {test_metrics['mae']:.3f}; R²: {test_metrics['r2']:.3f}")
print(f"Mean-baseline RMSE: {baseline_metrics['rmse']:.3f}; relative RMSE reduction: {improvement:.1f}%")
print(f"Development-fit RMSE: {train_metrics['rmse']:.3f}; five-fold CV RMSE: {best_row['cv_rmse_mean']:.3f}")
print("A small train/test difference alone cannot prove absence of overfitting.")

**Interpretation limits:** this is one fixed holdout; CV fold SD is not a confidence interval. Inspect the measured errors
by ring group before claiming equal accuracy for young and old abalones. Continuous ring predictions are left unrounded
for regression evaluation. These estimates do not establish accuracy on new populations or invalid measurements.

**Stage 2 handoff:** move loading/validation, feature extraction, pipeline construction, and evaluation
into modules. Preserve the input schema, split seed, data checksum, and fitted preprocessing when serving predictions.
Keep `data/` out of Git. Repository hooks intentionally strip notebook outputs; rerun locally to view figures and tables.

Compare the selected model's development-fit error with its CV error to assess the generalization gap.
Low training error alone is not evidence of good generalization. Any later parameter changes should use development CV;
reusing this inspected test set for tuning would require a new independent final evaluation.

**Previously inspected holdout:** the seed-42 test set was evaluated in the earlier notebook version.
This revision selects the model using development CV, but reusing that holdout does not restore a strictly untouched final test.
Treat the test metrics as a reference benchmark; use fresh independent data for a confirmatory assessment.